In [3]:
import pandas as pd

df = pd.read_csv("lggsn_dataset.csv")
print("N =", len(df))
print(df["query"].value_counts())
print(df["success"].value_counts())


N = 8
query
chips          3
tennis         3
tomato soup    1
sauce\         1
Name: count, dtype: int64
success
1    4
0    4
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["success"]
)
print("train size:", len(train_df), "val size:", len(val_df))
print("train success counts:\n", train_df["success"].value_counts())
print("val success counts:\n", val_df["success"].value_counts())


train size: 6 val size: 2
train success counts:
 success
1    3
0    3
Name: count, dtype: int64
val success counts:
 success
0    1
1    1
Name: count, dtype: int64


Epoch 00 | train loss=0.6832 acc=0.50 | val loss=0.7518 acc=0.00
Epoch 01 | train loss=0.6633 acc=0.67 | val loss=0.7719 acc=0.00
Epoch 02 | train loss=0.6477 acc=0.67 | val loss=0.7894 acc=0.00
Epoch 03 | train loss=0.6330 acc=0.83 | val loss=0.8123 acc=0.00
Epoch 04 | train loss=0.6204 acc=0.83 | val loss=0.8383 acc=0.00
Epoch 05 | train loss=0.6037 acc=0.83 | val loss=0.8624 acc=0.00
Epoch 06 | train loss=0.5873 acc=0.83 | val loss=0.8915 acc=0.00
Epoch 07 | train loss=0.5733 acc=0.83 | val loss=0.9237 acc=0.00
Epoch 08 | train loss=0.5583 acc=0.83 | val loss=0.9580 acc=0.00
Epoch 09 | train loss=0.5390 acc=0.83 | val loss=0.9968 acc=0.00
Epoch 10 | train loss=0.5229 acc=0.83 | val loss=1.0417 acc=0.00
Epoch 11 | train loss=0.5047 acc=0.83 | val loss=1.0932 acc=0.00
Epoch 12 | train loss=0.4887 acc=0.83 | val loss=1.1512 acc=0.00
Epoch 13 | train loss=0.4682 acc=0.83 | val loss=1.2186 acc=0.00
Epoch 14 | train loss=0.4532 acc=0.83 | val loss=1.2920 acc=0.00
Epoch 15 | train loss=0.4321 acc=0.83 | val loss=1.3688 acc=0.00
Epoch 16 | train loss=0.4185 acc=0.83 | val loss=1.4503 acc=0.00
Epoch 17 | train loss=0.4022 acc=0.83 | val loss=1.5335 acc=0.00
Epoch 18 | train loss=0.3849 acc=0.83 | val loss=1.6134 acc=0.00
Epoch 19 | train loss=0.3704 acc=0.83 | val loss=1.7003 acc=0.00
With only 8 labeled samples (6 train, 2 val), the simple LG-GSN MLP reaches ~0.83 train accuracy but 0.0 val accuracy, indicating overfitting and that current evaluation is not meaningful yet. At this stage the goal is just to verify the end-to-end training pipeline works.

In [1]:
import pandas as pd

df = pd.read_csv("grasp_6dof/dataset/all_lggsn.csv")
print(df.shape)
print(df.columns)
print(df.head())做一个“小实验表”——几何 vs 多模态 Ablation


(4190, 14)
Index(['x', 'y', 'z', 'roll', 'pitch', 'yaw', 'width', 'score', 'dz',
       'dz_lift', 'need_dz', 'H', 'label', 'query'],
      dtype='object')
          x         y         z      roll     pitch       yaw     width  \
0  0.382042  0.002631  0.082374  0.045849 -0.006342  1.570796  0.022943   
1  0.382042  0.002631  0.082374  0.036544 -0.028412  2.094021  0.022943   
2  0.382042  0.002631  0.082374  0.017441 -0.042876  2.617765  0.022943   
3  0.382042  0.002631  0.082374 -0.006349 -0.045848 -3.141302  0.022943   
4  0.382042  0.002631  0.082374 -0.028431 -0.036529 -2.617329  0.022943   

      score   dz  dz_lift  need_dz     H  label     query  
0  1.005177  0.0      0.0      0.0  0.08      0  cylinder  
1  1.005177  0.0      0.0      0.0  0.08      0  cylinder  
2  1.005177  0.0      0.0      0.0  0.08      0  cylinder  
3  1.005177  0.0      0.0      0.0  0.08      0  cylinder  
4  1.005177  0.0      0.0      0.0  0.08      1  cylinder  


In [16]:
import numpy as np
import glob

sem_paths = sorted(glob.glob("semantic_pc/*.npz"))
print("N semantic_pc:", len(sem_paths))

sem_by_sid = {}   # sid -> list of (pc, query)
for p in sem_paths:
    d = np.load(p, allow_pickle=True)
    pc = d["pc"]                         # (N,3)
    q = str(d["query"]).strip().lower()
    meta = d["metas"].item()
    sid = meta["sid"]                    # 和 all_lggsn.csv 里的 sid 对齐

    sem_by_sid.setdefault(sid, []).append((pc, q))
print("N distinct sid with semantic_pc:", len(sem_by_sid))


N semantic_pc: 95
N distinct sid with semantic_pc: 11


In [17]:
import torch
from torch.utils.data import Dataset
import numpy as np
import random

class MultimodalGraspDataset(Dataset):
    def __init__(self, df, sem_by_sid, feat_cols, text_vocab=None):
        """
        df: all_lggsn.csv 读出来的 DataFrame
        sem_by_sid: {sid: [(pc, query), ...]} ，来自 semantic_pc
        feat_cols: 几何特征用到的列名列表（直接抄你 train_lggsn.py 里的那份）
        text_vocab: 可选，把 query 映射到一个 id，用来做简单的文本 embedding
        """
        self.sem_by_sid = sem_by_sid
        self.feat_cols = feat_cols

        # 1) 自动寻找一个类似 'sid' / 'scene' 的列名
        sid_col = None
        for c in df.columns:
            lc = c.lower()
            if "sid" in lc or "scene" in lc:
                sid_col = c
                break
        self.sid_col = sid_col

        if self.sid_col is not None and len(sem_by_sid) > 0:
            print(f"[INFO] Using '{self.sid_col}' as sid column.")
            valid_sids = set(sem_by_sid.keys())
            self.df = df[df[self.sid_col].isin(valid_sids)].reset_index(drop=True)
        else:
            print("[WARN] No sid/scene column found in df; will NOT align by sid.")
            self.df = df.reset_index(drop=True)

        # 2) 文本词典（简单版本：每个不同 query 一个 id）
        if text_vocab is None:
            qs = sorted(set(str(q).strip().lower()
                            for vals in sem_by_sid.values()
                            for _, q in vals))
            self.text2id = {q: i for i, q in enumerate(qs)}
        else:
            self.text2id = text_vocab

        # 预先把所有 sid 列表存起来，方便 fallback 随机选
        self._all_sids = list(sem_by_sid.keys())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1) 几何特征
        vals = row[self.feat_cols].to_numpy(dtype=np.float32)  # 强制转成 float32
        geom = torch.from_numpy(vals)                          # (Dg,)

        # 2) label (0/1)
        label_col = "label" if "label" in self.df.columns else "success"
        y = torch.tensor(row[label_col], dtype=torch.float32)

        # 3) 找对应的 semantic_pc
        if self.sid_col is not None:
            sid = row[self.sid_col]
            pcs = self.sem_by_sid.get(sid)
            if pcs is None:
                # 理论上不会到这里，兜底一下
                sid = random.choice(self._all_sids)
                pcs = self.sem_by_sid[sid]
        else:
            # 没法对齐，就随机选一个 sid 的样本（先保证 pipeline 能跑）
            sid = random.choice(self._all_sids)
            pcs = self.sem_by_sid[sid]

        pc, q = random.choice(pcs)          # 随机选这一 sid 下的一条语义点云
        # 3.1 固定点数
        MAX_POINTS = 1024
        num_points = pc.shape[0]
        if num_points >= MAX_POINTS:
            idx = np.random.choice(num_points, MAX_POINTS, replace=False)
        else:
            idx = np.random.choice(num_points, MAX_POINTS, replace=True)
        pc_fixed = pc[idx]
        pc = torch.from_numpy(pc_fixed.astype(np.float32))  # (MAX_POINTS, 3)

        # 4) 文本 id
        q_norm = str(q).strip().lower()
        tid = self.text2id.get(q_norm, 0)
        text_id = torch.tensor(tid, dtype=torch.long)

        return geom, pc, text_id, y


In [18]:
from torch.utils.data import DataLoader
from lggsn_multimodal import SemanticPointEncoder, TextEncoderPlaceholder, LggsnMultimodal

feature_cols = [
        "x", "y", "z",
        "roll", "pitch", "yaw",
        "width", "score",
        "dz", "dz_lift", "need_dz", "H",
    ]

dataset = MultimodalGraspDataset(df, sem_by_sid, feat_cols)
loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

text_vocab_size = len(dataset.text2id)
text_emb_dim = 512

txt_embedding = torch.nn.Embedding(text_vocab_size, text_emb_dim)
sem_enc = SemanticPointEncoder()
txt_enc = TextEncoderPlaceholder(in_dim=text_emb_dim, out_dim=256)
head = LggsnMultimodal(dim_geom=len(feat_cols), dim_sem=256, dim_text=256)

for geom, pc, text_id, y in loader:
    sem_feat = sem_enc(pc)
    text_emb = txt_embedding(text_id)
    text_feat = txt_enc(text_emb)
    prob = head(geom, sem_feat, text_feat)
    print("geom:", geom.shape, "pc:", pc.shape, "text_id:", text_id.shape, "prob:", prob.shape, "y:", y.shape)
    break


[WARN] No sid/scene column found in df; will NOT align by sid.
geom: torch.Size([32, 12]) pc: torch.Size([32, 1024, 3]) text_id: torch.Size([32]) prob: torch.Size([32]) y: torch.Size([32])


In [6]:
print(df.shape)
print(df.columns.tolist())
df.head()


(4190, 14)
['x', 'y', 'z', 'roll', 'pitch', 'yaw', 'width', 'score', 'dz', 'dz_lift', 'need_dz', 'H', 'label', 'query']


,x,y,z,roll,pitch,yaw,width,score,dz,dz_lift,need_dz,H,label,query
0,0.382042,0.002631,0.082374,0.045849,-0.006342,1.570796,0.022943,1.005177,0.0,0.0,0.0,0.08,0,cylinder
1,0.382042,0.002631,0.082374,0.036544,-0.028412,2.094021,0.022943,1.005177,0.0,0.0,0.0,0.08,0,cylinder
2,0.382042,0.002631,0.082374,0.017441,-0.042876,2.617765,0.022943,1.005177,0.0,0.0,0.0,0.08,0,cylinder
3,0.382042,0.002631,0.082374,-0.006349,-0.045848,-3.141302,0.022943,1.005177,0.0,0.0,0.0,0.08,0,cylinder
4,0.382042,0.002631,0.082374,-0.028431,-0.036529,-2.617329,0.022943,1.005177,0.0,0.0,0.0,0.08,1,cylinder


In [19]:
import torch
import torch.nn as nn
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

sem_enc = sem_enc.to(device)
txt_embedding = txt_embedding.to(device)
txt_enc = txt_enc.to(device)
head = head.to(device)

# 我们 head 现在返回的是 prob ∈ (0,1)，所以用 BCELoss 即可
criterion = nn.BCELoss()
optimizer = Adam(
    list(sem_enc.parameters()) +
    list(txt_embedding.parameters()) +
    list(txt_enc.parameters()) +
    list(head.parameters()),
    lr=1e-3
)

EPOCHS = 5

for epoch in range(EPOCHS):
    sem_enc.train(); txt_embedding.train(); txt_enc.train(); head.train()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    for geom, pc, text_id, y in loader:
        geom = geom.to(device)               # (B, 12)
        pc = pc.to(device)                   # (B, 1024, 3)
        text_id = text_id.to(device)         # (B,)
        y = y.to(device)                     # (B,)

        optimizer.zero_grad()

        sem_feat = sem_enc(pc)               # (B, 256)
        text_emb = txt_embedding(text_id)    # (B, 512)
        text_feat = txt_enc(text_emb)        # (B, 256)

        prob = head(geom, sem_feat, text_feat)  # (B,)

        loss = criterion(prob, y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss) * y.size(0)
        preds = (prob >= 0.5).float()
        acc = (preds == y).float().mean()
        total_acc += float(acc) * y.size(0)
        total_n += y.size(0)

    print(f"Epoch {epoch:02d} | loss={total_loss/total_n:.4f} acc={total_acc/total_n:.2f}")


device = cuda
Epoch 00 | loss=0.3884 acc=0.85
Epoch 01 | loss=0.2640 acc=0.93
Epoch 02 | loss=0.2474 acc=0.93
Epoch 03 | loss=0.2295 acc=0.93
Epoch 04 | loss=0.2254 acc=0.93


In [20]:
import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

dim_geom = len(feat_cols)   # 12
dim_sem = 256
dim_text_emb = 512
dim_text = 256

# 划分 train / val
n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val = n_total - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val])
print("train N =", len(train_ds), "val N =", len(val_ds))

def make_loaders(batch_size=64):
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size,
                            shuffle=False, drop_last=False)
    return train_loader, val_loader


device = cuda
train N = 3352 val N = 838


In [21]:
from lggsn_multimodal import SemanticPointEncoder, TextEncoderPlaceholder, LggsnMultimodal

def run_experiment(name, use_sem=True, use_text=True,
                   epochs=5, lr=1e-3, batch_size=64):
    """
    use_sem = False: 忽略 semantic_pc 分支（置零）
    use_text = False: 忽略 text 分支（置零）
    """
    print(f"\n=== Experiment: {name} (use_sem={use_sem}, use_text={use_text}) ===")
    train_loader, val_loader = make_loaders(batch_size=batch_size)

    text_vocab_size = len(dataset.text2id)

    # 模块初始化
    sem_enc = SemanticPointEncoder().to(device)
    txt_embedding = nn.Embedding(text_vocab_size, dim_text_emb).to(device)
    txt_enc = TextEncoderPlaceholder(in_dim=dim_text_emb, out_dim=dim_text).to(device)
    head = LggsnMultimodal(dim_geom=dim_geom,
                           dim_sem=dim_sem,
                           dim_text=dim_text).to(device)

    criterion = nn.BCELoss()
    optimizer = Adam(
        list(sem_enc.parameters()) +
        list(txt_embedding.parameters()) +
        list(txt_enc.parameters()) +
        list(head.parameters()),
        lr=lr
    )

    def forward_batch(geom, pc, text_id):
        geom = geom.to(device)
        pc = pc.to(device)
        text_id = text_id.to(device)

        # 语义点云特征
        sem_feat = sem_enc(pc)                   # (B, 256)
        # 文本特征
        text_emb = txt_embedding(text_id)        # (B, 512)
        text_feat = txt_enc(text_emb)            # (B, 256)

        # ablation: 关掉某些模态
        if not use_sem:
            sem_feat = torch.zeros_like(sem_feat)
        if not use_text:
            text_feat = torch.zeros_like(text_feat)

        prob = head(geom, sem_feat, text_feat)   # (B,)
        return prob

    best_val_acc = 0.0

    for epoch in range(epochs):
        # ---------- train ----------
        sem_enc.train(); txt_embedding.train(); txt_enc.train(); head.train()
        train_loss = 0.0
        train_correct = 0
        train_n = 0

        for geom, pc, text_id, y in train_loader:
            y = y.to(device)
            optimizer.zero_grad()

            prob = forward_batch(geom, pc, text_id)
            loss = criterion(prob, y)
            loss.backward()
            optimizer.step()

            train_loss += float(loss) * y.size(0)
            preds = (prob >= 0.5).float()
            train_correct += (preds == y).float().sum().item()
            train_n += y.size(0)

        train_loss /= train_n
        train_acc = train_correct / train_n

        # ---------- val ----------
        sem_enc.eval(); txt_embedding.eval(); txt_enc.eval(); head.eval()
        val_loss = 0.0
        val_correct = 0
        val_n = 0
        with torch.no_grad():
            for geom, pc, text_id, y in val_loader:
                y = y.to(device)
                prob = forward_batch(geom, pc, text_id)
                loss = criterion(prob, y)

                val_loss += float(loss) * y.size(0)
                preds = (prob >= 0.5).float()
                val_correct += (preds == y).float().sum().item()
                val_n += y.size(0)

        val_loss /= val_n
        val_acc = val_correct / val_n
        best_val_acc = max(best_val_acc, val_acc)

        print(f"Epoch {epoch:02d} | "
              f"train loss={train_loss:.4f} acc={train_acc:.2f} | "
              f"val loss={val_loss:.4f} acc={val_acc:.2f}")

    print(f"[{name}] best val acc = {best_val_acc:.3f}")
    return best_val_acc


In [22]:
res_geom_only = run_experiment(
    name="GeomOnly",
    use_sem=False,
    use_text=False,
    epochs=5,
    lr=1e-3
)

res_geom_sem = run_experiment(
    name="Geom+Semantic",
    use_sem=True,
    use_text=False,
    epochs=5,
    lr=1e-3
)

res_full = run_experiment(
    name="Geom+Semantic+Text",
    use_sem=True,
    use_text=True,
    epochs=5,
    lr=1e-3
)

print("\n=== Summary (best val acc) ===")
print(f"GeomOnly          : {res_geom_only:.3f}")
print(f"Geom+Semantic     : {res_geom_sem:.3f}")
print(f"Geom+Sem+Text     : {res_full:.3f}")



=== Experiment: GeomOnly (use_sem=False, use_text=False) ===
Epoch 00 | train loss=0.3762 acc=0.91 | val loss=0.2446 acc=0.93
Epoch 01 | train loss=0.2019 acc=0.93 | val loss=0.1954 acc=0.93
Epoch 02 | train loss=0.1600 acc=0.94 | val loss=0.1670 acc=0.94
Epoch 03 | train loss=0.1462 acc=0.95 | val loss=0.1524 acc=0.94
Epoch 04 | train loss=0.1336 acc=0.95 | val loss=0.1550 acc=0.95
[GeomOnly] best val acc = 0.949

=== Experiment: Geom+Semantic (use_sem=True, use_text=False) ===
Epoch 00 | train loss=0.3625 acc=0.91 | val loss=0.2529 acc=0.92
Epoch 01 | train loss=0.2318 acc=0.93 | val loss=0.2190 acc=0.93
Epoch 02 | train loss=0.1828 acc=0.94 | val loss=0.1926 acc=0.94
Epoch 03 | train loss=0.1533 acc=0.95 | val loss=0.1602 acc=0.94
Epoch 04 | train loss=0.1481 acc=0.95 | val loss=0.1493 acc=0.95
[Geom+Semantic] best val acc = 0.951

=== Experiment: Geom+Semantic+Text (use_sem=True, use_text=True) ===
Epoch 00 | train loss=0.4639 acc=0.81 | val loss=0.2878 acc=0.93
Epoch 01 | train l